In [2]:
# extract comments with keywords
import json
import re
import time
import os
from IPython.display import clear_output

in_path  = r"D:\ERP\code\RC_2019-04\RC_2019-04"  # path of original data
out_path = r"F:\ERP_data\filtered_MSJG_RC.jsonl"

# M: Mathematics
M = re.compile(
    r"\b("
    r"maths|"
    r"mathematics|"
    r"algebra|"
    r"calculus|"
    r"geometry|"
    r"math\s+(class|major|degree|department|education)"
    r")\b",
    re.I
)

# S: STEM
S = re.compile(
    r"\b("
    r"stem|"
    r"engineering|"
    r"physics|"
    r"engineer|"
    r"computer\s*science|"
    r"cs\s*major|"
    r"programming|"
    r"coding|"
    r"coder|"
    r"software\s*engineer|"
    r"software\s*developer|"
    r"data\s*science|"
    r"data\s*scientist|"
    r"machine\s*learning|"
    r"artificial\s*intelligence|"
    r"women\s*in\s*tech|"
    r"girls\s*who\s*code"
    r")\b",
    re.I
)

# G: Gender
G = re.compile(
    r"\b("
    r"gender|"
    r"sexism|"
    r"sexist|"
    r"discrimination|"
    r"women|"
    r"woman|"
    r"girls|"
    r"female|"
    r"underrepresented|"
    r"male\s*dominated|"
    r"gender\s*gap"
    r")\b",
    re.I
)

report_every_lines = 400000
report_every_seconds = 15

def context_filter(text):

    text_lower = text.lower()


    sentences = re.split(
        r'[.!?]+',
        text_lower
    )


    # sentence level
    for sent in sentences:

        if (
            (M.search(sent) or S.search(sent))
            and G.search(sent)
        ):
            return True


    # comment level
    if (
        (M.search(text_lower) or S.search(text_lower))
        and G.search(text_lower)
    ):
        return True


    return False

# Helper functions
def format_bytes(num_bytes):
    num_bytes = float(num_bytes)
    for unit in ["B", "KB", "MB", "GB", "TB"]:
        if num_bytes < 1024:
            return f"{num_bytes:.2f} {unit}"
        num_bytes /= 1024
    return f"{num_bytes:.2f} PB"

def format_seconds(seconds):
    if seconds == float("inf"):
        return "inf"
    seconds = int(seconds)
    h = seconds // 3600
    m = (seconds % 3600) // 60
    s = seconds % 60
    return f"{h:d}:{m:02d}:{s:02d}"

# Init
total_size = os.path.getsize(in_path)

scanned_bytes = 0
scanned_lines = 0
kept = 0

start_time = time.time()
last_report_time = start_time

clear_output(wait=True)
print("Start filtering...\n")
print(f"Scanned: 0.00 GB / {format_bytes(total_size)} (0.00%)")
print("Scanned lines: 0")
print("Kept lines: 0")
print("Speed: 0.00 MB/s")
print("ETA: calculating...")

# Main loop
with open(in_path, "rb") as fin, open(out_path, "wb") as fout:
    for raw_line in fin:
        scanned_lines += 1
        scanned_bytes += len(raw_line)
        try:
            line = raw_line.decode("utf-8", errors="replace")
            obj = json.loads(line)
        except Exception:
            continue

        body = obj.get("body") or ""

        if body in ("[deleted]", "[removed]"):
            pass
        elif "I am a bot" in body or "AUTOMOD" in body:
            pass
        elif context_filter(body):
            fout.write(raw_line)
            kept += 1

        now = time.time()
        need_report = (
            scanned_lines % report_every_lines == 0
            or (now - last_report_time) >= report_every_seconds
        )

        if need_report:
            elapsed = now - start_time
            pct = (scanned_bytes / total_size) * 100 if total_size else 0
            speed = scanned_bytes / elapsed if elapsed > 0 else 0
            eta = (total_size - scanned_bytes) / speed if speed > 0 else float("inf")

            clear_output(wait=True)
            print("Filtering in progress...\n")
            print(
                f"Scanned: {format_bytes(scanned_bytes)} / {format_bytes(total_size)} "
                f"({pct:.2f}%)"
            )
            print(f"Scanned lines: {scanned_lines:,}")
            print(f"Kept lines: {kept:,}")
            print(f"Speed: {format_bytes(speed)}/s")
            print(f"ETA: {format_seconds(eta)}")

            last_report_time = now
            
# Final summary
elapsed = time.time() - start_time
pct = (scanned_bytes / total_size) * 100 if total_size else 0
out_size = os.path.getsize(out_path) if os.path.exists(out_path) else 0
ratio = (out_size / total_size) * 100 if total_size else 0
avg_speed = scanned_bytes / elapsed if elapsed > 0 else 0

clear_output(wait=True)
print("Done.\n")
print(
    f"Scanned: {format_bytes(scanned_bytes)} / {format_bytes(total_size)} "
    f"({pct:.2f}%)"
)
print(f"Scanned lines: {scanned_lines:,}")
print(f"Kept lines: {kept:,}")
print(f"Output size: {format_bytes(out_size)}")
print(f"Output/Input ratio: {ratio:.4f}%")
print(f"Average speed: {format_bytes(avg_speed)}/s")
print(f"Saved: {out_path}")
print(f"Total time: {format_seconds(elapsed)}")

Done.

Scanned: 151.71 GB / 151.71 GB (100.00%)
Scanned lines: 138,473,643
Kept lines: 13,431
Output size: 30.70 MB
Output/Input ratio: 0.0198%
Average speed: 20.49 MB/s
Saved: F:\ERP_data\filtered_MSJG_RC.jsonl
Total time: 2:06:22


In [1]:
# extract 200 samples
import json
import random
import pandas as pd

file_path = r"F:\ERP_data\filtered_MSJG_RC.jsonl"
output_path = r"F:\ERP_data\manual_check_sample_7_28.xlsx"

# Reservoir sampling
sample_size = 200
samples = []

with open(file_path, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        obj = json.loads(line)
        item = {
            "body": obj.get("body", ""),
            "subreddit": obj.get("subreddit", "")
        }
        if i < sample_size:
            samples.append(item)
        else:
            j = random.randint(0, i)
            if j < sample_size:
                samples[j] = item
                
# Create dataframe
df = pd.DataFrame(samples)
df["relevant"] = ""
df["reason"] = ""
df.to_excel(
    output_path,
    index=False
)

print("Done")
print("Sample size:", len(df))
print("Saved:", output_path)

df.head()

Done
Sample size: 200
Saved: F:\ERP_data\manual_check_sample_7_28.xlsx


,body,subreddit,relevant,reason
0,Good for you on being invited to attend the co...,cscareerquestions,,
1,He just wants you as a fuck buddy or a number ...,dating,,
2,Coincidentally I’ve been meeting a lot of peop...,INTP,,
3,Well his job could be automated by artificial ...,politics,,
4,Full contents: \n\n**Prices**\n* $1 minimum f...,GameDeals,,


In [1]:
# classifier
import pandas as pd
import json
import os

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

from sklearn.model_selection import cross_val_score
from sklearn.metrics import classification_report

train_path = r"F:\ERP_data\manual_check_sample_7_28.xlsx"
train_df = pd.read_excel(train_path)
train_df.head()

,id,body,subreddit,relevant,reason
0,1,Good for you on being invited to attend the co...,cscareerquestions,1,Gender-STEM narrative
1,2,He just wants you as a fuck buddy or a number ...,dating,0,No relevant STEM gender context
2,3,Coincidentally I’ve been meeting a lot of peop...,INTP,0,No relevant STEM gender context
3,4,Well his job could be automated by artificial ...,politics,0,No relevant STEM gender context
4,5,Full contents: \n\n**Prices**\n* $1 minimum f...,GameDeals,0,No relevant STEM gender context


In [3]:
#TF-IDF transforming
X_text = train_df["body"].fillna("")
y = train_df["relevant"]

vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1,2),
    min_df=2,
    max_features=10000
)

X_train = vectorizer.fit_transform(X_text)

print(X_train.shape)

(200, 2813)


In [4]:
#train Logistic Regression
clf = LogisticRegression(
    class_weight="balanced",
    max_iter=1000,
    random_state=42
)

clf.fit(
    X_train,
    y
)

LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)

In [5]:
# cross-validation
scores = cross_val_score(
    clf,
    X_train,
    y,
    cv=5,
    scoring="f1"
)

print(scores)
print(
    "Mean F1:",
    scores.mean()
)

[0.8        0.6        0.75675676 0.82051282 0.57142857]
Mean F1: 0.7097396297396296


In [6]:
# read candidate corpus
candidate_path = r"F:\ERP_data\filtered_MSJG_RC.jsonl"
texts = []
objects = []

with open(candidate_path,"r",encoding="utf-8") as f:
    for line in f:
        obj = json.loads(line)
        body = obj.get("body","")
        texts.append(body)
        objects.append(obj)
print("Candidate size:", len(texts))

Candidate size: 13431


In [7]:
#predict candidate
X_new = vectorizer.transform(texts)
prob = clf.predict_proba(X_new)[:,1]
pred = clf.predict(X_new)
pd.Series(prob).describe()

count    13431.000000
mean         0.495296
std          0.093247
min          0.233798
25%          0.428090
50%          0.482744
75%          0.556925
max          0.854848
dtype: float64

In [8]:
pd.Series(prob).quantile(
    [0.5,0.7,0.8,0.9]
)

0.5    0.482744
0.7    0.539290
0.8    0.577021
0.9    0.624687
dtype: float64

In [9]:
# TOP 20 HIGH PROBABILITY
top_idx = prob.argsort()[-20:][::-1]
print("===== TOP 20 HIGH PROBABILITY =====")
for i in top_idx:
    print("\nPROB:", round(prob[i], 3))
    print(objects[i]["body"][:300])
    print("-"*80)

===== TOP 20 HIGH PROBABILITY =====

PROB: 0.855
But what can women do *in STEM* that men cannot?
--------------------------------------------------------------------------------

PROB: 0.848
Which women’s dorms are STEM focused?
--------------------------------------------------------------------------------

PROB: 0.824
MOAR WOMEN IN STEM!!!!!!!!
--------------------------------------------------------------------------------

PROB: 0.824
Women in STEM are all, collectively, my hero
--------------------------------------------------------------------------------

PROB: 0.824
Yay for women in STEM!
--------------------------------------------------------------------------------

PROB: 0.824
I have been assured there are no women in stem!
--------------------------------------------------------------------------------

PROB: 0.824
DAE WOMEN STEM?
--------------------------------------------------------------------------------

PROB: 0.824
Yay, women in STEM!
---------------------------

In [10]:
# TOP 20 LOW PROBABILITY
bottom_idx = prob.argsort()[:20]
print("===== TOP 20 LOW PROBABILITY =====")
for i in bottom_idx:
    print("\nPROB:", round(prob[i], 3))
    print(objects[i]["body"][:300])
    print("-"*80)

===== TOP 20 LOW PROBABILITY =====

PROB: 0.234
&gt;Yes he provided examples of their attacks. Booker is a supporter of M4A and other progressive policies. 

He most certainly did not.  He used an article about Warren's DNA, but the author is a native American, who's tribe had denounced what Warren did, and she had other articles in other publica
--------------------------------------------------------------------------------

PROB: 0.238
Impersonator could not capture how spectacular some of Trump's quotes are:

&gt; Look, having nuclear — my uncle was a great professor and scientist and engineer, Dr. John Trump at MIT; good genes, very good genes, OK, very smart, the Wharton School of Finance, very good, very smart — you know, if y
--------------------------------------------------------------------------------

PROB: 0.239
&gt;IVANKA is standing with her back to the door, hair somewhat disheveled and makeup starting to wear off or run. 

&gt;TRUMP is standing with back to the window

In [11]:
#test for threshold=0.6
for t in [0.55,0.6,0.65]:
    print(
        "threshold:",
        t,
        "kept:",
        (prob>=t).sum()
    )

threshold: 0.55 kept: 3620
threshold: 0.6 kept: 1932
threshold: 0.65 kept: 883


In [12]:
#extract comments with p>=0.6
threshold = 0.6
final_path = r"F:\ERP_data\final_corpus_06.jsonl"

count = 0
with open(final_path, "w", encoding="utf-8") as fout:
    for obj, p in zip(objects, prob):
        if p >= threshold:
            obj["classifier_probability"] = float(p)
            fout.write(
                json.dumps(
                    obj,
                    ensure_ascii=False
                ) + "\n"
            )
            count += 1
print("Saved:", final_path)
print("Final corpus size:", count)

Saved: F:\ERP_data\final_corpus_06.jsonl
Final corpus size: 1932


In [16]:
# 50samples from 1932 comments
threshold = 0.6
final_indices = [
    i for i, p in enumerate(prob)
    if p >= threshold
]

print(len(final_indices))

import random
import pandas as pd
sample_indices = random.sample(
    final_indices,
    50)
validation = []

for i in sample_indices:
    validation.append({
        "body": objects[i]["body"],
        "probability": prob[i],
        "subreddit": objects[i].get("subreddit","")
    })
df_validation = pd.DataFrame(validation)
df_validation.to_excel(
    r"F:\ERP_data\corpusRC06_50sample.xlsx",
    index=False
)
print("Saved")

1932
Saved


In [17]:
# filter the pure URLs
import json
import re

file_path = r"F:\ERP_data\final_corpusRC_06.jsonl"
url_pattern = re.compile(
    r"^(https?://\S+|www\.\S+)$",
    re.I
)

total = 0
pure_url = 0
short_30 = 0
short_50 = 0
short_100 = 0

url_examples = []
short_examples = []
with open(file_path, "r", encoding="utf-8") as f:
    for line in f:
        obj = json.loads(line)
        body = obj.get("body", "").strip()
        total += 1
        # pure URL
        if url_pattern.match(body):
            pure_url += 1
            if len(url_examples) < 5:
                url_examples.append(body)
        # length statistics
        if len(body) < 30:
            short_30 += 1
            if len(short_examples) < 5:
                short_examples.append(body)
        if len(body) < 50:
            short_50 += 1
        if len(body) < 100:
            short_100 += 1
print("Total comments:", total)
print("\nPure URL:")
print(
    pure_url,
    f"({pure_url/total:.2%})"
)

print("\nLength <30:")
print(
    short_30,
    f"({short_30/total:.2%})"
)

print("\nLength <50:")
print(
    short_50,
    f"({short_50/total:.2%})"
)

print("\nLength <100:")
print(
    short_100,
    f"({short_100/total:.2%})"
)

print("\nPure URL examples:")
for x in url_examples:
    print("-", x)
print("\nShort examples:")
for x in short_examples:
    print("-", x)

Total comments: 1932

Pure URL:
2 (0.10%)

Length <30:
14 (0.72%)

Length <50:
53 (2.74%)

Length <100:
172 (8.90%)

Pure URL examples:
- https://www.theatlantic.com/science/archive/2018/02/the-more-gender-equality-the-fewer-women-in-stem/553592/
- https://www.theatlantic.com/science/archive/2018/02/the-more-gender-equality-the-fewer-women-in-stem/553592/

Short examples:
- Do you mean women in STEM?
- Yay for women in STEM!
- DAE WOMEN STEM?
- DAE stem woman?
- MOAR WOMEN IN STEM!!!!!!!!
